# 04 - Phase 4: QLoRA fine-tuning of the answer-writing model

> Run with **Save Version -> Save & Run All (Commit)**. Settings: **Accelerator = GPU T4 x2**, **Internet = On**.
> A commit run is limited to 12 h; the notebook stops training cleanly before that (`TRAIN_HOURS`).

**Inputs (Add Input):** the Phase 0 dataset (`train_processed.csv`), the Phase 2 output dataset (`question_neighbours.json`, `chunk_hits.json`),
and - only for the `full` mode - the chunks dataset (`chunks.jsonl`).

**Three variants (the ablation for the report)**

| mode | prompt | role in the report |
|---|---|---|
| `none` | question only | B2: plain fine-tuning, no retrieval |
| `nn` | question + 2 nearest training Q&A pairs | proposed model without PDFs |
| `full` | `nn` + source-page excerpts (pages cited by the neighbours) | proposed model with PDFs |

`RUN` below chooses which modes to train in this session. Two modes train **in parallel, one per GPU**. Suggested order:
first `["nn", "none"]`, then in a second session `["full", ...]`, and finally the winning mode on the full training set.

**Loss is computed on the answer tokens only.** 20% of examples get a distractor reference (RAFT-style) so the model learns to ignore irrelevant context.

In [ ]:
!pip install -q -U peft bitsandbytes accelerate huggingface_hub

In [ ]:
%%writefile prompts.py
"""Phase 4: build the training / inference prompts.

Modes
  none : question only                                   (baseline B2: plain fine-tuning)
  nn   : question + the 2 nearest training Q&A pairs      (proposed model without PDFs)
  full : nn + source-page excerpts                         (proposed model with PDFs)

Neighbours for TRAIN questions already exclude the question's own group (see Phase 2), so the model is trained to
synthesise an answer from related-but-different answers, not to copy an identical one.
Pure ASCII source on purpose.
"""
import json
import random

SYSTEM = (
    "You are an expert Indian agricultural advisor. Answer the farmer's question in English with a detailed, "
    "well-structured advisory: key facts or symptoms, thresholds where relevant, cultural, biological and chemical "
    "management with exact doses and units, and safety notes. Use the reference material when it applies to this "
    "question; never invent doses."
)


def trim_words(text, n):
    w = text.split()
    return " ".join(w[:n]) + (" ..." if len(w) > n else "")


def trim_tokens(text, n, tok):
    """Cut text to n tokens. Token budgets (not word counts) keep every prompt inside the length limit: numeric-heavy
    text can use twice as many tokens per word as prose."""
    ids = tok(text, add_special_tokens=False)["input_ids"]
    return text if len(ids) <= n else tok.decode(ids[:n]) + " ..."


def build_user(mode, question, refs, excerpts, ref_words=300, pdf_words=180, tok=None, ref_tokens=380, pdf_tokens=200):
    """refs: list of (question, answer); excerpts: list of (label, text). With tok given, trimming is by tokens."""
    def cut(text, words, tokens):
        return trim_tokens(text, tokens, tok) if tok is not None else trim_words(text, words)
    if mode == "none":
        return "Question: " + question
    parts = ["Reference answers to similar questions:"]
    for i, (q, a) in enumerate(refs, 1):
        parts.append("[%d] Q: %s\nA: %s" % (i, q, cut(a, ref_words, ref_tokens)))
    if mode == "full" and excerpts:
        parts.append("Source excerpts:")
        for label, t in excerpts:
            parts.append("[%s] %s" % (label, cut(t, pdf_words, pdf_tokens)))
    parts.append("Question: " + question)
    return "\n\n".join(parts)


class Context:
    """Holds the lookups needed to build prompts for any query set."""

    def __init__(self, rows, page_text=None, chunk_text=None):
        self.qid2row = {r["question_id"]: r for r in rows}
        self.rows = rows
        self.page_text = page_text or {}      # (doc_id, page) -> first chunk text of that page
        self.chunk_text = chunk_text or []    # chunk id -> text
        self._cit = {}

    def citations(self, qid):
        if qid not in self._cit:
            self._cit[qid] = json.loads(self.qid2row[qid]["citations_json"])
        return self._cit[qid]

    def propagated_pages(self, neighbour_ids, limit=2):
        """Pages cited by the nearest training questions, in neighbour order. Returns [(label, text)]."""
        out, seen = [], set()
        for nid in neighbour_ids:
            for c in self.citations(nid):
                if c.get("kind") != "zoho":
                    continue
                for p in c.get("pages", []):
                    key = (c["doc_id"], p)
                    if key in self.page_text and key not in seen:
                        seen.add(key)
                        name = (c.get("name") or "source")[:60]
                        out.append(("%s, p.%d" % (name, p), self.page_text[key]))
                        if len(out) == limit:
                            return out
        return out


def make_example(ctx, mode, qid, question, neighbours, chunk_hits=None, answer=None, noise=0.0, rng=None,
                 pool_qids=None, n_refs=2, tok=None):
    """neighbours: [[question_id, score], ...] best first."""
    rng = rng or random.Random(0)
    nids = [n[0] for n in neighbours]
    refs_ids = nids[:n_refs]
    if noise and pool_qids and len(refs_ids) == n_refs and rng.random() < noise:
        refs_ids[-1] = rng.choice(pool_qids)      # RAFT-style distractor: the model learns to ignore bad context
    refs = [(ctx.qid2row[i]["question"], ctx.qid2row[i]["final_answer"]) for i in refs_ids]
    excerpts = []
    if mode == "full":
        excerpts = ctx.propagated_pages(nids[:3])
        if len(excerpts) < 2 and chunk_hits and ctx.chunk_text:
            for cid in chunk_hits:
                if len(excerpts) == 2:
                    break
                excerpts.append(("retrieved passage", ctx.chunk_text[cid]))
        if noise and excerpts and pool_qids and rng.random() < noise:
            other = ctx.propagated_pages([rng.choice(pool_qids)], limit=1)
            if other:
                excerpts[-1] = other[0]
    ex = {"qid": qid, "mode": mode, "system": SYSTEM, "user": build_user(mode, question, refs, excerpts, tok=tok)}
    if answer is not None:
        ex["answer"] = answer
    return ex

In [ ]:
%%writefile train_qlora.py
"""QLoRA fine-tuning with prompt-masked loss (only the answer tokens are trained on).

Uses the plain transformers Trainer + peft (no TRL) so it does not depend on TRL's changing API.
Run one process per GPU:  CUDA_VISIBLE_DEVICES=0 python train_qlora.py --train t.jsonl --dev d.jsonl --out dir

Safety nets for long Kaggle runs: a wall-clock limit (--train_hours) that saves the adapter and stops cleanly,
a NaN guard, checkpoints every --save_steps with automatic resume, and a tokens/sec estimate for planning.
--tiny / --no_quant exist only for CPU smoke tests. Pure ASCII source on purpose.
"""
import argparse
import inspect
import json
import math
import os
import random
import time

import torch
from transformers import (AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer,
                          TrainerCallback, TrainingArguments)
from peft import LoraConfig, get_peft_model

TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


def parse():
    p = argparse.ArgumentParser()
    p.add_argument("--train", required=True)
    p.add_argument("--dev", default="")
    p.add_argument("--out", required=True)
    p.add_argument("--model", default="Qwen/Qwen2.5-3B-Instruct")
    p.add_argument("--max_len", type=int, default=2560)
    p.add_argument("--epochs", type=float, default=1.0)
    p.add_argument("--max_steps", type=int, default=-1)
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument("--bs", type=int, default=1)
    p.add_argument("--grad_accum", type=int, default=16)
    p.add_argument("--lora_r", type=int, default=16)
    p.add_argument("--lora_alpha", type=int, default=32)
    p.add_argument("--save_steps", type=int, default=100)
    p.add_argument("--eval_steps", type=int, default=100)
    p.add_argument("--train_hours", type=float, default=10.0)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--optim", default="paged_adamw_8bit")
    p.add_argument("--no_quant", action="store_true")
    p.add_argument("--tiny", action="store_true")
    p.add_argument("--no_grad_ckpt", action="store_true")
    p.add_argument("--ddp_backend", default="", help="CPU smoke tests only (gloo); GPUs use NCCL automatically")
    return p.parse_args()


def encode_example(tok, ex, max_len, im_end_id):
    """Chat-format the prompt, append the answer, and mask the prompt in the labels. None if the prompt leaves no room."""
    msgs = [{"role": "system", "content": ex["system"]}, {"role": "user", "content": ex["user"]}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    prompt = tok(text, add_special_tokens=False)["input_ids"]
    ans = tok(ex["answer"], add_special_tokens=False)["input_ids"]
    room = max_len - len(prompt)
    if room < 256:
        return None
    if len(ans) + 1 > room:
        ans = ans[:room]  # truncated: no end token, so the model is never taught to stop mid-sentence
        truncated = True
    else:
        ans = ans + [im_end_id]
        truncated = False
    return {"input_ids": prompt + ans, "labels": [-100] * len(prompt) + ans, "truncated": truncated,
            "n_prompt": len(prompt)}


def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")]


class ListDataset(torch.utils.data.Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return {"input_ids": self.items[i]["input_ids"], "labels": self.items[i]["labels"]}


class Collator:
    def __init__(self, pad_id):
        self.pad_id = pad_id

    def __call__(self, feats):
        m = max(len(f["input_ids"]) for f in feats)
        ids = torch.full((len(feats), m), self.pad_id, dtype=torch.long)
        lab = torch.full((len(feats), m), -100, dtype=torch.long)
        att = torch.zeros((len(feats), m), dtype=torch.long)
        for i, f in enumerate(feats):
            n = len(f["input_ids"])
            ids[i, :n] = torch.tensor(f["input_ids"])
            lab[i, :n] = torch.tensor(f["labels"])
            att[i, :n] = 1
        return {"input_ids": ids, "labels": lab, "attention_mask": att}


def _dist():
    return torch.distributed.is_available() and torch.distributed.is_initialized()


class Guard(TrainerCallback):
    """Wall-clock limit + NaN guard. On a stop, a checkpoint is saved (so the next session resumes) and the adapter is
    written. With several processes (DDP) the stop decision is all-reduced, so every rank stops on the same step."""

    def __init__(self, hours):
        self.t_end = time.time() + hours * 3600
        self.reason = "completed"
        self.nan = False
        self.start_step = 0

    def on_train_begin(self, args, state, control, **kw):
        self.start_step = state.global_step      # > 0 when resumed from a checkpoint

    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs and (math.isnan(logs["loss"]) or math.isinf(logs["loss"])):
            self.nan = True
        return control

    def on_step_end(self, args, state, control, **kw):
        flag = 2 if self.nan else (1 if time.time() > self.t_end else 0)
        if _dist():
            t = torch.tensor([flag], device="cuda" if torch.cuda.is_available() else "cpu")
            torch.distributed.all_reduce(t, op=torch.distributed.ReduceOp.MAX)
            flag = int(t.item())
        if flag:
            self.reason = "nan_loss" if flag == 2 else "time_limit"
            if flag == 2:
                print("NaN/inf loss - stopping. Try a smaller model or lower lr.", flush=True)
            else:
                control.should_save = True               # checkpoint for the next session
            control.should_training_stop = True
        return control


def build_model(a, tok):
    cuda = torch.cuda.is_available()
    if a.tiny:
        cfg = AutoConfig.for_model("qwen2", vocab_size=len(tok), hidden_size=64, intermediate_size=128,
                                   num_hidden_layers=2, num_attention_heads=4, num_key_value_heads=2,
                                   max_position_embeddings=4096, tie_word_embeddings=True)
        return AutoModelForCausalLM.from_config(cfg)
    if a.no_quant:
        return AutoModelForCausalLM.from_pretrained(a.model, torch_dtype=torch.float16 if cuda else torch.float32)
    from peft import prepare_model_for_kbit_training
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)
    rank = int(os.environ.get("LOCAL_RANK", 0))       # one full 4-bit copy per GPU under DDP
    model = AutoModelForCausalLM.from_pretrained(a.model, quantization_config=bnb, torch_dtype=torch.float16,
                                                 device_map={"": rank}, attn_implementation="sdpa")
    return prepare_model_for_kbit_training(model, use_gradient_checkpointing=not a.no_grad_ckpt,
                                           gradient_checkpointing_kwargs={"use_reentrant": False})


def main():
    a = parse()
    random.seed(a.seed)
    torch.manual_seed(a.seed)
    os.makedirs(a.out, exist_ok=True)
    cuda = torch.cuda.is_available()
    tok = AutoTokenizer.from_pretrained(a.model)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    im_end_id = tok.convert_tokens_to_ids("<|im_end|>")
    assert im_end_id is not None and im_end_id != tok.unk_token_id, "tokenizer has no <|im_end|> (use a Qwen instruct model)"

    t0 = time.time()
    train_raw = load_jsonl(a.train)
    enc = [e for e in (encode_example(tok, ex, a.max_len, im_end_id) for ex in train_raw) if e]
    avg_len = sum(len(e["input_ids"]) for e in enc) / max(1, len(enc))
    print("train examples: %d kept of %d | truncated answers: %d | avg tokens %.0f | max %d | tokenised in %.0fs" % (
        len(enc), len(train_raw), sum(e["truncated"] for e in enc), avg_len, max(len(e["input_ids"]) for e in enc),
        time.time() - t0), flush=True)
    dev_ds = None
    if a.dev:
        dev_enc = [e for e in (encode_example(tok, ex, a.max_len, im_end_id) for ex in load_jsonl(a.dev)) if e]
        dev_ds = ListDataset(dev_enc)
        print("dev examples:", len(dev_enc), flush=True)

    model = build_model(a, tok)
    model.config.use_cache = False
    if a.no_quant and not a.tiny and not a.no_grad_ckpt:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(r=a.lora_r, lora_alpha=a.lora_alpha, lora_dropout=0.05,
                                             target_modules=TARGETS, task_type="CAUSAL_LM"))
    for p in model.parameters():  # fp16 training needs fp32 trainable weights (GradScaler cannot unscale fp16 grads)
        if p.requires_grad:
            p.data = p.data.float()
    model.print_trainable_parameters()

    sig = inspect.signature(TrainingArguments.__init__).parameters
    kw = dict(output_dir=a.out, per_device_train_batch_size=a.bs, per_device_eval_batch_size=1,
              gradient_accumulation_steps=a.grad_accum, learning_rate=a.lr, lr_scheduler_type="cosine",
              num_train_epochs=a.epochs, max_steps=a.max_steps, logging_steps=5,
              save_steps=a.save_steps, save_total_limit=2, fp16=cuda, optim=a.optim, report_to="none",
              remove_unused_columns=False, dataloader_num_workers=0, seed=a.seed, max_grad_norm=1.0,
              disable_tqdm=True, gradient_checkpointing=(cuda and not a.no_grad_ckpt and not a.no_quant))
    if dev_ds is not None:
        kw["eval_strategy" if "eval_strategy" in sig else "evaluation_strategy"] = "steps"
        kw["eval_steps"] = a.eval_steps
    if "gradient_checkpointing_kwargs" in sig:
        kw["gradient_checkpointing_kwargs"] = {"use_reentrant": False}
    world = int(os.environ.get("WORLD_SIZE", 1))
    if world > 1:
        kw["ddp_find_unused_parameters"] = False
    if a.ddp_backend:
        kw["ddp_backend"] = a.ddp_backend
    # 3% warm-up: old transformers has warmup_ratio; new transformers takes a float in warmup_steps as the ratio
    kw["warmup_ratio" if "warmup_ratio" in sig else "warmup_steps"] = 0.03
    dropped = sorted(k for k in kw if k not in sig)
    if dropped:
        print("WARNING: this transformers version has no TrainingArguments option(s):", dropped, "- ignoring them", flush=True)
        kw = {k: v for k, v in kw.items() if k in sig}
    guard = Guard(a.train_hours)
    trainer = Trainer(model=model, args=TrainingArguments(**kw), train_dataset=ListDataset(enc), eval_dataset=dev_ds,
                      data_collator=Collator(tok.pad_token_id), callbacks=[guard])
    resume = any(d.startswith("checkpoint-") for d in os.listdir(a.out))
    t1 = time.time()
    trainer.train(resume_from_checkpoint=True if resume else None)
    elapsed = time.time() - t1
    final = os.path.join(a.out, "final")
    trainer.save_model(final)                      # main process only (Trainer handles it)
    if not trainer.is_world_process_zero():
        return
    tok.save_pretrained(final)
    steps = trainer.state.global_step
    run_steps = steps - guard.start_step
    per_step = elapsed / max(1, run_steps)
    ex_per_step = a.bs * a.grad_accum * world
    summary = {"status": guard.reason, "steps": steps, "steps_this_session": run_steps, "resumed_from_step": guard.start_step,
               "world_size": world, "elapsed_s": round(elapsed, 1), "sec_per_step": round(per_step, 2),
               "examples_per_step": ex_per_step, "avg_tokens": round(avg_len),
               "tokens_per_sec_est": round(avg_len * ex_per_step / max(1e-9, per_step)),
               "n_train": len(enc), "truncated_answers": sum(e["truncated"] for e in enc),
               "steps_per_epoch": len(enc) // ex_per_step, "max_len": a.max_len,
               "loss_log": [(h["step"], round(h["loss"], 4)) for h in trainer.state.log_history if "loss" in h],
               "eval_log": [(h["step"], round(h["eval_loss"], 4)) for h in trainer.state.log_history if "eval_loss" in h]}
    json.dump(summary, open(os.path.join(a.out, "train_summary.json"), "w"), indent=1)
    print("DONE", {k: v for k, v in summary.items() if k not in ("loss_log", "eval_log")}, flush=True)


if __name__ == "__main__":
    main()

## 0. Configuration

In [ ]:
import glob, os, sys, json, gzip, csv, random, time, subprocess, collections as C
csv.field_size_limit(10 ** 9)
_missing = [f for f in ("prompts.py", "train_qlora.py") if not os.path.exists(f)]
assert not _missing, "missing " + str(_missing) + " - run the %%writefile cells at the top first (use Run All, do not start from a lower cell)"
sys.path.insert(0, os.getcwd())

import torch
if not os.environ.get("P4_EXTRA"):   # (the env var is a CPU smoke-test hook, never set on Kaggle)
    assert torch.cuda.is_available(), ("No GPU attached to this session. Right panel -> Session options -> Accelerator -> "
                                       "GPU T4 x2 (the session restarts), then Run All from the top.")
    print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

MODEL = os.environ.get("P4_MODEL", "Qwen/Qwen2.5-3B-Instruct")
RUN = ["nn", "none"]          # modes to train in this session (see the table above); later e.g. ["full"]
N_TRAIN = 4000                # training questions per run (same subset for every mode). Final run: 0 = all 12,825
EPOCHS = 1.0
MAX_LEN = 2560                # prompt + answer tokens
GRAD_ACCUM = 16               # effective batch = 16 examples
NOISE = 0.2                   # share of examples with a distractor reference
N_DEV = 60                    # dev questions used for the eval-loss curve
TRAIN_HOURS = 8.0             # wall-clock limit per run; the adapter is saved when it is reached
EXTRA = os.environ.get("P4_EXTRA", "").split()   # debug hook only (CPU smoke tests); empty on Kaggle

def find(pattern, required=True):
    hit = glob.glob("/kaggle/input/**/" + pattern, recursive=True)
    if not hit and required:
        print("not found:", pattern, "- files under /kaggle/input (first 40):")
        for i, (dp, dn, fn) in enumerate(os.walk("/kaggle/input")):
            for x in fn[:8]:
                print("  ", os.path.join(dp, x))
            if i > 40:
                break
        raise AssertionError("attach a dataset containing " + pattern)
    return hit[0] if hit else None

def op(path):
    return gzip.open(path, "rt", encoding="utf-8") if path.endswith(".gz") else open(path, encoding="utf-8")

TRAIN_CSV, NN_PATH = find("train_processed*.csv"), find("question_neighbours.json*")
HITS_PATH, CHUNKS = find("chunk_hits.json*", required=False), find("chunks.jsonl*", required="full" in RUN)
WORK = "/kaggle/working"
print("modes:", RUN, "| model:", MODEL, "| N_TRAIN:", N_TRAIN or "all")

## 1. Build the prompts (CPU)

In [ ]:
import prompts as P
rows = list(csv.DictReader(open(TRAIN_CSV, encoding="utf-8")))
NN = json.load(op(NN_PATH))
hits = json.load(op(HITS_PATH)) if (HITS_PATH and "full" in RUN) else {}
page_text, chunk_text = {}, []
if "full" in RUN:
    for line in op(CHUNKS):
        c = json.loads(line)
        chunk_text.append(c["text"])
        page_text.setdefault((c["doc_id"], c["page"]), c["text"])   # first chunk of each page
    print("chunks:", len(chunk_text), "| pages with text:", len(page_text))
ctx = P.Context(rows, page_text, chunk_text)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)   # used to trim context by TOKENS so no prompt exceeds MAX_LEN

rng = random.Random(42)
train_rows = [r for r in rows if r["split"] == "train"]
dev_rows = [r for r in rows if r["split"] == "dev"][:N_DEV]
sub = train_rows if not N_TRAIN else rng.sample(train_rows, min(N_TRAIN, len(train_rows)))
pool_qids = [r["question_id"] for r in train_rows]
print("train subset:", len(sub), "| dev:", len(dev_rows))

def build(mode, items, split, noise):
    r = random.Random(1)
    out = []
    for row in items:
        q = row["question_id"]
        out.append(P.make_example(ctx, mode, q, row["question"], NN[split].get(q, []) if mode != "none" else [],
                                  chunk_hits=hits.get(split, {}).get(q), answer=row["final_answer"], noise=noise,
                                  rng=r, pool_qids=pool_qids, tok=tok))
    return out

os.makedirs(WORK + "/data", exist_ok=True)
for mode in RUN:
    for name, items, split, noise in (("train", sub, "train", NOISE if mode != "none" else 0.0), ("dev", dev_rows, "dev", 0.0)):
        ex = build(mode, items, split, noise)
        with open("%s/data/%s_%s.jsonl" % (WORK, name, mode), "w", encoding="utf-8") as f:
            for e in ex:
                f.write(json.dumps(e, ensure_ascii=False) + "\n")
    lens = [len(tok(e["system"] + e["user"], add_special_tokens=False)["input_ids"]) for e in ex]
    print("mode %-5s | prompt tokens: mean %.0f, max %d (limit leaves %d for the answer)" % (mode, sum(lens) / len(lens), max(lens), MAX_LEN - max(lens)))
ex = build(RUN[0], dev_rows[:1], "dev", 0.0)[0]
print("\n--- example prompt (mode %s) ---\n%s\n--- target answer starts: %s" % (RUN[0], ex["user"][:1800], ex["answer"][:200].replace("\n", " ")))

## 2. Download the model once (so the two parallel runs do not race on the download)

In [ ]:
if not EXTRA:
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL)
import torch
N_GPU = torch.cuda.device_count()
print("GPUs:", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])

## 3. Pilot - 3 optimizer steps to measure speed and memory before the long run
If this fails with an out-of-memory error, lower `MAX_LEN` (e.g. 2048). If the loss is `nan`, switch `MODEL` to `Qwen/Qwen2.5-1.5B-Instruct`.

In [ ]:
def cmd(mode, out, extra=()):
    return [sys.executable, "train_qlora.py", "--train", "%s/data/train_%s.jsonl" % (WORK, mode), "--dev", "%s/data/dev_%s.jsonl" % (WORK, mode),
            "--out", out, "--model", MODEL, "--max_len", str(MAX_LEN), "--grad_accum", str(GRAD_ACCUM),
            "--epochs", str(EPOCHS), "--train_hours", str(TRAIN_HOURS)] + list(extra) + EXTRA

mode = RUN[0]
r = subprocess.run(cmd(mode, WORK + "/pilot", ["--max_steps", "3", "--grad_accum", "4", "--save_steps", "1000", "--eval_steps", "1000"]),
                   env=dict(os.environ, CUDA_VISIBLE_DEVICES="0", TOKENIZERS_PARALLELISM="false"), capture_output=True, text=True)
print(r.stdout[-1800:]); print(r.stderr[-1500:] if r.returncode else "")
assert r.returncode == 0, "pilot failed - read the error above"
s = json.load(open(WORK + "/pilot/train_summary.json"))
per_example = s["sec_per_step"] / 4
n = len(open("%s/data/train_%s.jsonl" % (WORK, mode)).readlines())
proj = per_example * n * EPOCHS / 3600
print("pilot: %.1f s per example (~%d tokens/s) -> %d examples x %.1f epochs ~ %.1f h per run" % (per_example, s["tokens_per_sec_est"], n, EPOCHS, proj))
if proj > TRAIN_HOURS:
    print("WARNING: projected time exceeds TRAIN_HOURS=%.1f - the run will stop early. Reduce N_TRAIN or EPOCHS." % TRAIN_HOURS)
import shutil; shutil.rmtree(WORK + "/pilot", ignore_errors=True)

## 4. Train - one process per GPU (waves of N_GPU runs)

In [ ]:
os.makedirs(WORK + "/logs", exist_ok=True)
slots = max(1, N_GPU)
for w in range(0, len(RUN), slots):
    wave = RUN[w:w + slots]
    procs = []
    for gpu, mode in enumerate(wave):
        log = open("%s/logs/%s.log" % (WORK, mode), "w")
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), TOKENIZERS_PARALLELISM="false")
        procs.append((mode, subprocess.Popen(cmd(mode, "%s/adapters/%s" % (WORK, mode)), env=env, stdout=log, stderr=subprocess.STDOUT)))
        print("started", mode, "on GPU", gpu)
    t0 = time.time()
    while any(p.poll() is None for _, p in procs):
        time.sleep(120)
        for mode, p in procs:
            lines = [l for l in open("%s/logs/%s.log" % (WORK, mode), errors="ignore").read().splitlines()
                     if any(k in l for k in ("loss", "DONE", "Error", "error", "Traceback", "examples", "trainable"))]
            print("[%4.0f min] %-5s %s" % ((time.time() - t0) / 60, mode, lines[-1][:170] if lines else "(no output yet)"), flush=True)
    for mode, p in procs:
        print("finished", mode, "exit code", p.returncode)

## 5. Results - training and dev loss, and whether each run finished cleanly

In [ ]:
for mode in RUN:
    f = "%s/adapters/%s/train_summary.json" % (WORK, mode)
    if not os.path.exists(f):
        print(mode, ": NO SUMMARY - see logs/%s.log" % mode); continue
    s = json.load(open(f))
    print("%-5s status=%s steps=%d (%.1f h) | train loss %s -> %s | dev loss %s" % (mode, s["status"], s["steps"], s["elapsed_s"] / 3600,
          s["loss_log"][0][1] if s["loss_log"] else "-", s["loss_log"][-1][1] if s["loss_log"] else "-", s["eval_log"]))

## 6. Sanity check - does the adapter write a sensible answer? (3 dev questions per mode)

In [ ]:
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
if EXTRA:
    print("skipped in CPU smoke test")
else:
    tok = AutoTokenizer.from_pretrained(MODEL)
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    for mode in RUN:
        base = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, torch_dtype=torch.float16, device_map={"": 0})
        model = PeftModel.from_pretrained(base, "%s/adapters/%s/final" % (WORK, mode)).eval()
        for e in [json.loads(l) for l in open("%s/data/dev_%s.jsonl" % (WORK, mode))][:3]:
            text = tok.apply_chat_template([{"role": "system", "content": e["system"]}, {"role": "user", "content": e["user"]}], tokenize=False, add_generation_prompt=True)
            ids = tok(text, return_tensors="pt", add_special_tokens=False).to("cuda:0")
            with torch.no_grad():
                out = model.generate(**ids, max_new_tokens=350, do_sample=False, repetition_penalty=1.05)
            gen = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
            print("=" * 80, "\n[%s] Q: %s" % (mode, e["user"].split("Question: ")[-1][:150]), "\n", gen[:700], "\n... (%d words generated)" % len(gen.split()))
        del model, base; gc.collect(); torch.cuda.empty_cache()

## After it finishes
Output tab -> **New Dataset** (e.g. `nlp-qna-adapters`): it contains `adapters/<mode>/final` (the LoRA weights), the training summaries and logs.
Paste me the **results table (section 5)** and the **sample answers (section 6)**. Phase 5 then generates answers for the 1,500 hold-out rows
and the 1,000 unseen questions and scores every system with the shared evaluation harness.